# Paper main figure: one return-curve panel per run of this notebook

Pulls `eval/return` from W&B for one environment, fills missing evaluation points the same way
as the `*_interpolated_padded.xlsx` workbooks, smooths with the same EMA, and saves **one panel
PDF** plus standalone legend PDFs. Six panels are arranged 2 rows x 3 columns inside the ICLR
single-column text width (`\textwidth` = 5.5 in). Every panel of a figure gets the **same plot area**
(`plot_area()`, derived from `ROW_N`, `N_YLABEL`, `PANEL_H`); a panel file is only as wide as the labels it
shows, so a row pays for the y label once and the saved space goes to every plot. Include the PDFs at
their native size (no `width=`) with `\hfill` between them.

Axes follow the paper rules: x = **Environment Steps** = episodes x the env's max episode length
(nominal: Metaworld episodes that end on success are counted at full length, so say so in the
caption), ending exactly at the training budget with a labelled last tick; y = **Avg. Return**
(`eval/return` is the mean over one evaluation round of `eval_episodes` episodes).

Workflow per environment: edit the **config cell**, run all cells, check the per-seed quality
table, adjust `MAX_EPISODES` if a run should be cut, re-run the plot cell.


In [1]:
import json
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.patheffects as pe
from matplotlib.ticker import FuncFormatter, MaxNLocator


from paper_style import set_paper_style   # shared style: font sizes live in paper_style.py only


set_paper_style()


## Config (the only cell to edit per environment)

Run names follow the sweep convention `{env}_{model}_h{h}_l{l}_s{seed}_v2`. `METHODS` order is
the legend / drawing order. Style rule: **colored solid = memory model, neutral dashed =
reference bound** (Oracle sees the context; Markov has no memory).

**Fixed paper palette and legend order** (every figure, every environment): MATE `#E41A1C` (red, ours),
GPT-2 `#009E73` (green), LSTM `#0072B2` (blue), SplAgger `#AA3377` (red-purple: LSTM recurrence + MATE
aggregation), Mamba `#E69F00` (orange), Oracle black dashed, Markov `#999999` gray dashed. All colored
pairs pass the dataviz CVD check (min protan/deutan ΔE 10.1) and the normal-vision floor (min ΔE 15.7,
MATE–SplAgger; MATE's thicker line separates them further). A method without runs in an environment is
simply not drawn; keep it in `METHODS` so the legend order never changes. `EXCLUDE` leaves a method out of
this panel on purpose (e.g. no Oracle for Vehicle Racing): it is neither fetched nor drawn, the shared
`legend_{orientation}` files still list every method, and a panel-specific
`legend_{PROJECT}_{orientation}` without the excluded ones is saved as well.

`INCLUDE_RUNNING = True` draws W&B runs that are still `running` up to the last evaluation **all** of a
method's included seeds have reached (preview only; set `False` for the camera-ready figure).


In [2]:
ENTITY = "mate_research"
PROJECT = "carl_vehicle_racing_all"              # W&B project = registered env string (see main.py wandb.init)
ENV_PREFIX = "carl"               # {env} in RUN_NAME_TEMPLATE
RUN_NAME_TEMPLATE = "{env}_{model}_s{seed}_v2"
SEEDS = [0, 1, 2, 3, 4]
METHOD_SEEDS = {}                    # per-method seed override, e.g. {"MATE": [0, 2]}; others use SEEDS

METHODS = {                          # label -> run-name fields; order = legend order (fixed across the paper)
    "MATE":     dict(model="mate",     h=256, l=2),
    "GPT-2":    dict(model="gpt",      h=256, l=2),
    "LSTM":     dict(model="lstm",     h=256, l=2),
    "SplAgger": dict(model="splagger", h=256, l=2),
    "Mamba":    dict(model="mamba",    h=256, l=1),
    "Oracle":   dict(model="oracle",   h=512, l=2),
    "Markov":   dict(model="markov",   h=512, l=2),
}
STYLE = {                            # fixed paper palette (see the markdown above)
    "MATE":     dict(color="#E41A1C", ls="-"),   # ours: vivid red, drawn thicker and on top (EMPHASIS)
    "GPT-2":    dict(color="#009E73", ls="-"),   # Transformer: green
    "LSTM":     dict(color="#0072B2", ls="-"),   # RNN: blue
    "SplAgger": dict(color="#AA3377", ls="-"),   # red-purple, between LSTM blue and MATE red
    "Mamba":    dict(color="#E69F00", ls="-"),   # SSM: orange
    "Oracle":   dict(color="#000000", ls="--"),
    "Markov":   dict(color="#999999", ls="--"),
}
EXCLUDE = {"Oracle"}                 # methods left out of THIS panel (not fetched, not drawn); set() for none

METRIC = "eval/return"
XLABEL = "Environment Steps"         # x = episodes x EPISODE_LEN (nominal env steps)
YLABEL = "Avg. Return"               # eval/return = mean over the eval_episodes of one evaluation round
TITLE = "Vehicle Racing"                # panel title; None for no title
SHOW_YLABEL = True                   # only the leftmost panel of each row needs the y label
SHOW_XLABEL = True
YLIM = None                          # e.g. (-200, 0)

MAX_EPISODES = None                  # None -> inferred from the data (printed); set e.g. 40000 to cut the x-range
HORIZON_EPISODES = None              # training budget in episodes (x-axis end); None -> last eval point rounded up to 1000
EPISODE_LEN = None                   # max episode length (episodes -> env steps); None -> max_episode_steps(PROJECT)
EMA_FRACTION = 0.03                  # EMA time constant 1/(1-decay) = 3% of the panel's evaluation points, so
                                     # every env is smoothed over the same fraction of its training (user, 2026-09-23)
EMA_DECAY = 0.95                     # fixed decay, used only when EMA_FRACTION is None (the xlsx workbooks use 0.9)
MISSING_FRACTION_LIMIT = 0.10        # xlsx: exclude a run missing >10% of its expected grid
MAX_TRAILING_MISSING = 1             # xlsx: at most 1 trailing point filled by hold-last, else excluded

# Panel layout: every panel of a figure gets the SAME plot area (axes box). A panel file is only as large
# as the labels it shows, so a row pays for the y label once (SHOW_YLABEL only on its leftmost panel) and
# every plot gets the saved space. Include the PDFs at their NATIVE size (no width=) with \hfill between.
TEXT_WIDTH = 5.5                     # ICLR \textwidth (in)
ROW_N, N_YLABEL = 2, 2               # panels per row / how many of them show a y label
                                     # (Vehicle Racing: 2, 2 = a half-width panel; main 2x3 grid: 3, 1)
ROW_GAP = 0.06                       # gap between neighbouring panels (in); the tick-label reserves add ~0.04
PANEL_H = 1.70                       # height of a panel WITH title and x label (in)  (Vehicle Racing 1.70; main grid 1.35)
PANEL_W = 2.64                       # width of a panel WITH y label (in); None -> ROW_N panels fill TEXT_WIDTH
                                     # (Vehicle Racing: 0.48\textwidth = 2.64, fits a half-page minipage / wrapfigure)
YTICK_RESERVE = "0000"               # widest y tick label in the figure: room reserved in every panel
XTICK_RESERVE_RIGHT = "8M"           # widest LAST x tick label (half of it overhangs the plot)
                                     # (main grid: the widest over its six panels, e.g. "125M")
OUTER_PAD = 0.02                     # in
LEGEND_IN_PANEL = dict(loc="lower right", ncol=2)   # ax.legend kwargs for a legend inside the panel; None -> none
LINE_W = 0.6                         # same as the main figure (overlapping curves stay distinguishable)
EMPHASIS = "MATE"                    # drawn last with LINE_W * EMPHASIS_SCALE
EMPHASIS_SCALE = 1.35
BAND_ALPHA = 0.15
ERROR = "ci95"                       # band / bar half-width, with each curve's OWN n of seeds:
                                     # "ci95" = t(0.975, n-1) * std/sqrt(n) (Student-t 95% CI) | "sem" | "std"
LEGEND_EDGE = "#d9d9d9"                    # legend box edge color; "black" for a hard frame, "none" for no edge
LEGEND_SHADOW = dict(size=1.5, alpha=0.35, layers=4)   # soft drop shadow (points, total darkness, blur steps); None -> none

CACHE_DIR = Path("rl_results") / PROJECT
FIG_DIR = Path("figures")
FORCE_REFRESH = False                # True -> ignore the CSV cache and re-download from W&B
UNFINISHED_MAX_AGE_H = 12            # a cached run that was NOT "finished" is re-downloaded once its cache is older
INCLUDE_RUNNING = False              # True: draw still-running runs up to the common last evaluation (preview)

# expected (seq_model.name, is_oracle) per model tag, for the identity check
EXPECTED_SEQ = {
    "mate": ("mate", False), "mate_proj": ("mate", False), "splagger": ("splagger", False),
    "gpt": ("gpt", False), "lstm": ("lstm", False), "gru": ("gru", False), "rnn": ("rnn", False),
    "mamba": ("mamba", False),
    "oracle": ("markov", True), "markov": ("markov", False),
}


## W&B fetch with a local CSV cache

Full unfiltered `scan_history()` (as in the workbooks); rows with the metric are kept, `x =
_step` (the Learner logs with `step = n_episodes_total`). Raw values are cached under
`rl_results/{PROJECT}/raw/` and never modified by the filling step.


In [3]:
def run_name_for(label, seed):
    return RUN_NAME_TEMPLATE.format(env=ENV_PREFIX, seed=seed, **METHODS[label])


def _identity_check(label, meta):
    exp_name, exp_oracle = EXPECTED_SEQ.get(METHODS[label]["model"], (None, None))
    problems = []
    if exp_name is not None and meta.get("seq_name") != exp_name:
        problems.append(f"seq_model.name={meta.get('seq_name')!r} (expected {exp_name!r})")
    if exp_oracle is not None and bool(meta.get("is_oracle")) != exp_oracle:
        problems.append(f"is_oracle={meta.get('is_oracle')} (expected {exp_oracle})")
    if meta.get("hidden_size") not in (None, METHODS[label]["h"]):
        problems.append(f"hidden_size={meta.get('hidden_size')} (expected {METHODS[label]['h']})")
    if problems:
        warnings.warn(f"[{meta['run_name']}] identity mismatch: " + "; ".join(problems))


def fetch_run(run_name, force=False):
    '''Return (DataFrame[Step, Return], meta dict) or (None, meta) when the run is not found.
    Same cache policy as final_return_vs_depth_vis.ipynb: finished runs are never re-downloaded, unfinished
    ones (and "missing" verdicts) once their cache is older than UNFINISHED_MAX_AGE_H hours.'''
    raw_dir = CACHE_DIR / "raw"
    raw_dir.mkdir(parents=True, exist_ok=True)
    csv_path, meta_path = raw_dir / f"{run_name}.csv", raw_dir / f"{run_name}.meta.json"
    if meta_path.exists() and not force:
        meta = json.loads(meta_path.read_text())
        age_h = (time.time() - meta_path.stat().st_mtime) / 3600
        fresh = UNFINISHED_MAX_AGE_H is None or age_h < UNFINISHED_MAX_AGE_H
        if meta.get("state") == "missing" and fresh:
            return None, meta
        if meta.get("state") != "missing" and csv_path.exists() and (meta.get("state") == "finished" or fresh):
            return pd.read_csv(csv_path), meta

    import wandb
    api = wandb.Api(timeout=120)
    runs = list(api.runs(f"{ENTITY}/{PROJECT}", filters={"display_name": run_name}))
    if not runs:
        warnings.warn(f"no W&B run named {run_name!r} in {ENTITY}/{PROJECT}")
        meta = {"run_name": run_name, "state": "missing", "checked_at": time.strftime("%Y-%m-%dT%H:%M:%S")}
        meta_path.write_text(json.dumps(meta, indent=2))    # cache the verdict; no CSV is written
        if csv_path.exists():
            csv_path.unlink()
        return None, meta
    if len(runs) > 1:                      # prefer a finished run, newest among those (same rule as prefetch_wandb_cache.py)
        runs.sort(key=lambda r: (r.state == "finished", str(r.created_at)))
        warnings.warn(f"{len(runs)} runs named {run_name!r}; using {runs[-1].id} (state={runs[-1].state}, newest finished)")
    run = runs[-1]
    cfg = run.config
    seq = cfg.get("config_seq", {}).get("seq_model", {})
    meta = {
        "run_name": run_name, "run_id": run.id, "state": run.state, "created_at": str(run.created_at),
        "eval_interval": cfg.get("config_env", {}).get("eval_interval"),
        "seq_name": seq.get("name"), "is_oracle": seq.get("is_oracle", False),
        "hidden_size": seq.get("hidden_size"), "n_layer": seq.get("n_layer"),
        "project_output": cfg.get("config_seq", {}).get("project_output"),
    }
    rows = [(row["_step"], row[METRIC]) for row in run.scan_history() if row.get(METRIC) is not None]
    df = pd.DataFrame(rows, columns=["Step", "Return"]).sort_values("Step").reset_index(drop=True)
    df.to_csv(csv_path, index=False)
    meta_path.write_text(json.dumps(meta, indent=2))
    return df, meta


## Grid filling and EMA (mirrors the workbook `Methodology` sheet)

* expected grid: `eval_interval, 2·eval_interval, …, MAX_EPISODES`
* run missing > `MISSING_FRACTION_LIMIT` of the grid → excluded
* internal gaps: linear interpolation in episode coordinates
* leading gaps: repeat the first observed value (`padded`)
* trailing gaps: ≤ `MAX_TRAILING_MISSING` → hold-last (`tail_extended`), otherwise excluded
* status precedence: `padded` > `tail_extended` > `interpolated` > `original` > `excluded`
* then EMA with `ema[0] = x[0]`, `ema[t] = decay·ema[t-1] + (1-decay)·x[t]`


In [4]:
def expected_grid(eval_interval, max_episodes):
    return np.arange(eval_interval, int(max_episodes) + 1, eval_interval)


def fill_to_grid(steps, values, grid, missing_fraction_limit=MISSING_FRACTION_LIMIT,
                 max_trailing_missing=MAX_TRAILING_MISSING):
    '''Returns (filled values on `grid` or None, status, info dict).'''
    steps = np.asarray(steps, dtype=np.int64)
    values = np.asarray(values, dtype=np.float64)
    on_grid = np.isin(steps, grid)
    n_off_grid = int((~on_grid).sum())
    obs = dict(zip(steps[on_grid], values[on_grid]))          # duplicates: last one wins
    present = np.array([s in obs for s in grid])
    n_expected, n_observed = len(grid), int(present.sum())
    info = dict(eval_points=n_observed, expected_eval_points=n_expected,
                n_missing=n_expected - n_observed, n_off_grid=n_off_grid,
                n_leading=0, n_trailing=0, n_internal=0, exclusion_reason="")
    if n_observed == 0:
        info["exclusion_reason"] = "no evaluations on the grid"
        return None, "excluded", info
    first, last = int(np.argmax(present)), int(len(grid) - 1 - np.argmax(present[::-1]))
    info["n_leading"], info["n_trailing"] = first, n_expected - 1 - last
    info["n_internal"] = info["n_missing"] - info["n_leading"] - info["n_trailing"]
    if info["n_missing"] / n_expected > missing_fraction_limit:
        info["exclusion_reason"] = f"missing fraction {info['n_missing'] / n_expected:.1%} > {missing_fraction_limit:.0%}"
        return None, "excluded", info
    if info["n_trailing"] > max_trailing_missing:
        info["exclusion_reason"] = f"{info['n_trailing']} trailing points missing > {max_trailing_missing}"
        return None, "excluded", info

    xs = grid[present]
    ys = np.array([obs[s] for s in xs])
    filled = np.empty(n_expected)
    filled[first:last + 1] = np.interp(grid[first:last + 1], xs, ys)   # internal linear interpolation
    filled[:first] = ys[0]                                            # leading: repeat first observed
    filled[last + 1:] = ys[-1]                                        # trailing: hold-last
    if info["n_leading"] > 0:
        status = "padded"
    elif info["n_trailing"] > 0:
        status = "tail_extended"
    elif info["n_internal"] > 0:
        status = "interpolated"
    else:
        status = "original"
    return filled, status, info


def ema_decay(n_points):
    """EMA decay whose time constant 1/(1-decay) is EMA_FRACTION of the n_points evaluations of a panel:
    Cheetah (195 evals) ~0.83, Hopper (390) ~0.91, ML (781) ~0.96, Ant/Walker (976) ~0.97."""
    if EMA_FRACTION is None:
        return EMA_DECAY
    return float(np.clip(1.0 - 1.0 / (EMA_FRACTION * n_points), 0.0, 0.999))


def ema(x, decay=EMA_DECAY):
    x = np.asarray(x, dtype=np.float64)
    out = np.empty_like(x)
    out[0] = x[0]
    for t in range(1, len(x)):
        out[t] = decay * out[t - 1] + (1.0 - decay) * x[t]
    return out


## Load every method, print the per-seed quality table

`MAX_EPISODES=None` infers the horizon as the largest grid point observed in any fetched run
(printed below). Pin it explicitly if you want to cut the x-range; runs are then judged against
that shorter grid.


In [5]:
def load_all(max_episodes=MAX_EPISODES, force=FORCE_REFRESH):
    drawn = [label for label in METHODS if label not in EXCLUDE]
    raw = {}                                            # (label, seed) -> (df, meta)
    for label in drawn:
        for seed in METHOD_SEEDS.get(label, SEEDS):
            raw[(label, seed)] = fetch_run(run_name_for(label, seed), force=force)

    intervals = {m["eval_interval"] for df, m in raw.values() if df is not None}
    assert len(intervals) == 1, f"eval_interval differs across runs: {intervals}"
    eval_interval = int(intervals.pop())
    if max_episodes is None:
        last_steps = [int(df["Step"].max()) for df, _ in raw.values() if df is not None and len(df)]
        max_episodes = (max(last_steps) // eval_interval) * eval_interval
        print(f"MAX_EPISODES inferred = {max_episodes} (eval_interval={eval_interval})")
    grid = expected_grid(eval_interval, max_episodes)

    results, rows = {}, []
    for label in drawn:
        curves = []
        for seed in METHOD_SEEDS.get(label, SEEDS):
            df, meta = raw[(label, seed)]
            if df is None:
                rows.append(dict(method=label, seed=seed, run_name=meta["run_name"], state=meta["state"],
                                 status="missing", included=False, eval_points=0,
                                 expected=len(grid), n_missing=len(grid), reason="run not found"))
                continue
            _identity_check(label, meta)
            running = INCLUDE_RUNNING and meta.get("state") == "running"
            run_grid = grid[grid <= int(df["Step"].max())] if running else grid
            filled, status, info = fill_to_grid(df["Step"].values, df["Return"].values, run_grid)
            if running and filled is not None:
                status += f" (running, up to {int(run_grid[-1])})"
            if info["n_off_grid"]:
                warnings.warn(f"[{meta['run_name']}] dropped {info['n_off_grid']} off-grid evaluation rows")
            rows.append(dict(method=label, seed=seed, run_name=meta["run_name"], state=meta["state"],
                             status=status, included=filled is not None, eval_points=info["eval_points"],
                             expected=info["expected_eval_points"], n_missing=info["n_missing"],
                             reason=info["exclusion_reason"]))
            if filled is not None:
                curves.append(ema(filled, ema_decay(len(grid))))
        if not curves:
            warnings.warn(f"{label}: no included seeds, it will not be drawn")
            continue
        n_t = min(len(c) for c in curves)               # running seeds: cut to the shortest one
        if n_t < len(grid):
            warnings.warn(f"{label}: drawn only up to {int(grid[n_t - 1])} episodes (running seeds)")
        per_seed = np.stack([c[:n_t] for c in curves])  # (n_seeds, T)
        results[label] = dict(
            grid=grid[:n_t], per_seed=per_seed, mean=per_seed.mean(0),
            std=per_seed.std(0, ddof=1) if len(per_seed) > 1 else np.zeros(n_t),
            n=len(per_seed),
        )
    quality = pd.DataFrame(rows)
    return results, quality


results, quality = load_all()
with pd.option_context("display.max_rows", 200, "display.width", 200):
    display(quality)


wandb: Currently logged in as: himchan00 (piggene00) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
/tmp/ipykernel_497581/2048889174.py:38: UserWarning: no W&B run named 'carl_mate_s3_v2' in mate_research/carl_vehicle_racing_all
  warnings.warn(f"no W&B run named {run_name!r} in {ENTITY}/{PROJECT}")
/tmp/ipykernel_497581/2048889174.py:38: UserWarning: no W&B run named 'carl_mate_s4_v2' in mate_research/carl_vehicle_racing_all
  warnings.warn(f"no W&B run named {run_name!r} in {ENTITY}/{PROJECT}")
/tmp/ipykernel_497581/2048889174.py:38: UserWarning: no W&B run named 'carl_gpt_s3_v2' in mate_research/carl_vehicle_racing_all
  warnings.warn(f"no W&B run named {run_name!r} in {ENTITY}/{PROJECT}")
/tmp/ipykernel_497581/2048889174.py:38: UserWarning: no W&B run named 'carl_gpt_s4_v2' in mate_research/carl_vehicle_racing_all
  warnings.warn(f"no W&B run named {run_name!r} in {ENTITY}/{PROJECT}")
/tmp/ipykernel_497581/2048889174.py:38: UserWarning: no W&B run named 'carl

MAX_EPISODES inferred = 39936 (eval_interval=256)


/tmp/ipykernel_497581/2048889174.py:38: UserWarning: no W&B run named 'carl_markov_s4_v2' in mate_research/carl_vehicle_racing_all
  warnings.warn(f"no W&B run named {run_name!r} in {ENTITY}/{PROJECT}")


,method,seed,run_name,state,status,included,eval_points,expected,n_missing,reason
0,MATE,0,carl_mate_s0_v2,finished,original,True,156,156,0,
1,MATE,1,carl_mate_s1_v2,finished,original,True,156,156,0,
2,MATE,2,carl_mate_s2_v2,finished,original,True,156,156,0,
3,MATE,3,carl_mate_s3_v2,missing,missing,False,0,156,156,run not found
4,MATE,4,carl_mate_s4_v2,missing,missing,False,0,156,156,run not found
5,GPT-2,0,carl_gpt_s0_v2,finished,original,True,156,156,0,
6,GPT-2,1,carl_gpt_s1_v2,finished,original,True,156,156,0,
7,GPT-2,2,carl_gpt_s2_v2,finished,original,True,156,156,0,
8,GPT-2,3,carl_gpt_s3_v2,missing,missing,False,0,156,156,run not found
9,GPT-2,4,carl_gpt_s4_v2,missing,missing,False,0,156,156,run not found


## Panel

Saved as `figures/{PROJECT}_return.pdf` (vector, Type-42 fonts) plus a PNG preview. The x axis runs
from 0 to the training budget (`HORIZON_EPISODES` x `EPISODE_LEN`) and its last tick is exactly the
budget, so the total number of training steps is always printed. No legend
inside the panel; the legends are separate files (next cell).


In [6]:
def _si_formatter(v, _pos):
    for div, suffix in ((1e9, "B"), (1e6, "M"), (1e3, "k")):
        if abs(v) >= div:
            return f"{v / div:g}{suffix}"
    return f"{v:g}"


def max_episode_steps(project):
    """Registered max_episode_steps of the env behind a W&B project (configs/envs/*.py)."""
    m = re.fullmatch(r"tmaze_(passive|active)_T-(\d+)", project)
    if m:
        return int(m[2]) + (1 if m[1] == "passive" else 3)     # passive: T+1, active: T+2*1+1
    if project in ("ML10", "ML45"):
        return 500                                              # config_env.max_episode_steps
    if project in ("cheetah-vel", "ant-dir", "hopper-param", "walker-param") or project.startswith("carl_"):
        return 200
    raise ValueError(f"no known episode length for {project!r}; set EPISODE_LEN")


def budget_tick_candidates(xmax):
    """Tick sets 0..xmax whose LAST tick is exactly xmax (the training budget), best first: a round step
    dividing xmax (densest first), then a round step whose last tick is replaced by / followed by xmax
    (e.g. T-Maze budgets = episodes x (T+1)), then [0, xmax/2, xmax], then [0, xmax]."""
    decade = 10.0 ** np.floor(np.log10(xmax))
    steps = [m * decade for m in (0.1, 0.2, 0.25, 0.5, 1, 2, 2.5, 5)]
    divides = [abs(xmax / s - round(xmax / s)) < 1e-6 for s in steps]
    for step, exact in zip(steps, divides):
        if exact and 3 <= round(xmax / step) <= 6:
            yield np.linspace(0, xmax, int(round(xmax / step)) + 1)
    for step, exact in zip(steps, divides):
        n = int(np.floor(xmax / step + 1e-9))
        if not exact and 2 <= n <= 6:
            ticks = list(np.arange(n + 1) * step)
            if xmax - ticks[-1] < 0.5 * step:
                ticks[-1] = xmax                               # too close to the budget to label both
            else:
                ticks.append(xmax)
            yield np.array(ticks)
    yield np.array([0, xmax / 2, xmax])
    yield np.array([0, xmax])


def set_budget_xticks(ax, xmax, min_gap_pt=1.0):
    """x axis 0..xmax with the densest candidate tick set whose labels do not collide (call after the
    labels/title are set: it draws the figure to measure the tick labels)."""
    ax.set_xlim(0, xmax)
    ax.xaxis.set_major_formatter(FuncFormatter(_si_formatter))
    fig = ax.figure
    gap = min_gap_pt * fig.dpi / 72
    for ticks in budget_tick_candidates(xmax):
        ax.set_xticks(ticks)
        fig.canvas.draw()
        boxes = sorted((t.get_window_extent() for t in ax.get_xticklabels() if t.get_text()), key=lambda b: b.x0)
        if all(a.x1 + gap <= b.x0 for a, b in zip(boxes, boxes[1:])):
            break
    ax.set_xlim(0, xmax)
    return ticks


def style_legend_frame(legend):
    '''Thin light edge plus a soft drop shadow (stacked offset copies with decreasing alpha; stays vector in PDF).'''
    frame = legend.get_frame()
    frame.set_linewidth(0.4)
    if LEGEND_SHADOW:
        n, size, alpha = LEGEND_SHADOW["layers"], LEGEND_SHADOW["size"], LEGEND_SHADOW["alpha"]
        effects = [pe.SimplePatchShadow(offset=(size * k / n, -size * k / n), shadow_rgbFace="black",
                                        alpha=alpha / n) for k in range(n, 0, -1)]
        frame.set_path_effects(effects + [pe.Normal()])


def legend_handles(labels):
    """Legend lines drawn like the curves (dashed reference bounds, thicker EMPHASIS), in `labels` order."""
    handles = []
    for label in labels:
        st = STYLE[label]
        is_ref = st["ls"] != "-"
        lw = LINE_W * 1.4 * (EMPHASIS_SCALE if label == EMPHASIS else 1.0)
        handles.append(Line2D([], [], color=st["color"], ls=st["ls"], lw=lw, label=label,
                              dashes=(3, 1.5) if is_ref else (None, None)))
    return handles


def _text_extent(s, size, rotation=0, weight="normal"):
    """(width, height) in inches of `s` at `size` pt with the current rcParams (0 for an empty string)."""
    if not s:
        return 0.0, 0.0
    fig = plt.figure()
    t = fig.text(0, 0, s, fontsize=size, rotation=rotation, fontweight=weight)
    fig.canvas.draw()
    bb = t.get_window_extent()
    plt.close(fig)
    return bb.width / fig.dpi, bb.height / fig.dpi


def panel_margins(show_ylabel=True, show_xlabel=True, title=True):
    """Decoration space (left, right, bottom, top) in inches around the plot area. Tick labels always get the
    room of the widest expected label (YTICK_RESERVE, XTICK_RESERVE_RIGHT), so the plot area sits at the same
    place in every panel; only the y label, x label and title add space, and only where they are shown."""
    rc, pt = matplotlib.rcParams, 1 / 72
    ytick_w, ytick_h = _text_extent(YTICK_RESERVE, rc["ytick.labelsize"])
    xtick_h = _text_extent("0", rc["xtick.labelsize"])[1]
    left = OUTER_PAD + ytick_w + (rc["ytick.major.size"] + rc["ytick.major.pad"]) * pt
    if show_ylabel:
        left += _text_extent(YLABEL, rc["axes.labelsize"], rotation=90)[0] + rc["axes.labelpad"] * pt
    bottom = OUTER_PAD + xtick_h + (rc["xtick.major.size"] + rc["xtick.major.pad"]) * pt
    if show_xlabel:
        bottom += _text_extent(XLABEL, rc["axes.labelsize"])[1] + rc["axes.labelpad"] * pt
    top = OUTER_PAD + ytick_h / 2                              # the top y tick label overhangs the plot
    if title:
        title_h = _text_extent("Ag", rc["axes.titlesize"], weight=rc["axes.titleweight"])[1]
        top = max(top, OUTER_PAD + title_h + rc["axes.titlepad"] * pt)
    right = OUTER_PAD + _text_extent(XTICK_RESERVE_RIGHT, rc["xtick.labelsize"])[0] / 2
    return left, right, bottom, top


def plot_area():
    """(width, height) in inches of the plot area shared by every panel of the figure: ROW_N panels (N_YLABEL
    of them with a y label) plus ROW_GAP gaps fill TEXT_WIDTH, and a panel with title + x label is PANEL_H."""
    l1, r, b, t = panel_margins(show_ylabel=True)
    l0 = panel_margins(show_ylabel=False)[0]
    if PANEL_W is not None:                             # fixed file width (panel with a y label)
        return PANEL_W - l1 - r, PANEL_H - b - t
    deco = N_YLABEL * l1 + (ROW_N - N_YLABEL) * l0 + ROW_N * r
    return (TEXT_WIDTH - (ROW_N - 1) * ROW_GAP - deco) / ROW_N, PANEL_H - b - t


def make_panel(show_ylabel=True, show_xlabel=True, title=True):
    """Figure whose plot area is exactly plot_area(); the file is only as large as the decorations it shows."""
    pw, ph = plot_area()
    left, right, bottom, top = panel_margins(show_ylabel, show_xlabel, title)
    w, h = left + pw + right, bottom + ph + top
    fig = plt.figure(figsize=(w, h))
    ax = fig.add_axes([left / w, bottom / h, pw / w, ph / h])
    return fig, ax


def check_panel_fits(fig, ax):
    """Warn when a label is larger than its reserve (it would be clipped at the file edge)."""
    fig.canvas.draw()
    bb, fb = ax.get_tightbbox(fig.canvas.get_renderer()), fig.bbox
    over = {"left": -bb.x0, "right": bb.x1 - fb.x1, "bottom": -bb.y0, "top": bb.y1 - fb.y1}
    over = {k: round(v / fig.dpi, 3) for k, v in over.items() if v > 0.5}
    if over:
        warnings.warn(f"labels exceed the panel by {over} in: widen YTICK_RESERVE / XTICK_RESERVE_RIGHT")
    return fig.get_size_inches()


def error_of(r):
    """Half-width of the band, from the curve's OWN number of included seeds n: "ci95" is the Student-t 95%
    confidence interval of the mean, t(0.975, n-1) * std/sqrt(n), so curves with fewer seeds (failed or
    excluded runs) get correspondingly wider bands; "sem" is std/sqrt(n); "std" the sample std (ddof=1)."""
    from scipy.stats import t as student_t
    n = max(int(r["n"]), 1)
    if ERROR == "std":
        return r["std"]
    sem = r["std"] / np.sqrt(n)
    return sem * student_t.ppf(0.975, n - 1) if ERROR == "ci95" and n > 1 else sem


def plot_panel(results, title=TITLE, show_xlabel=SHOW_XLABEL, show_ylabel=SHOW_YLABEL, ylim=YLIM,
               out_stem=None):
    fig, ax = make_panel(show_ylabel=show_ylabel, show_xlabel=show_xlabel, title=bool(title))
    ep_len = EPISODE_LEN or max_episode_steps(PROJECT)
    for z, label in enumerate(METHODS):
        if label not in results:
            continue
        r, st = results[label], STYLE[label]
        is_ref = st["ls"] != "-"
        emph = label == EMPHASIS
        zorder = 2 + (0 if is_ref else 1) + (1 if emph else 0) + z * 0.01
        x = r["grid"] * ep_len
        e = error_of(r)
        ax.fill_between(x, r["mean"] - e, r["mean"] + e,
                        color=st["color"], alpha=BAND_ALPHA, lw=0, zorder=zorder - 1)
        ax.plot(x, r["mean"], color=st["color"], ls=st["ls"],
                lw=LINE_W * (EMPHASIS_SCALE if emph else 1.0), zorder=zorder,
                dashes=(3, 1.5) if is_ref else (None, None))
    last_eval = max(r["grid"][-1] for r in results.values())
    budget = (HORIZON_EPISODES or int(np.ceil(last_eval / 1000) * 1000)) * ep_len
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
    if ylim is not None:
        ax.set_ylim(*ylim)
    if title:
        ax.set_title(title)
    if show_xlabel:
        ax.set_xlabel(XLABEL)
    if show_ylabel:
        ax.set_ylabel(YLABEL)
    ax.grid(True, ls="--", alpha=0.5)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    if LEGEND_IN_PANEL:
        kw = dict(frameon=True, fancybox=False, edgecolor=LEGEND_EDGE, facecolor="white", framealpha=1.0,
                  handlelength=1.8, handletextpad=0.5, columnspacing=1.0, borderpad=0.4, borderaxespad=0.5,
                  labelspacing=0.3)
        kw.update(LEGEND_IN_PANEL)
        legend = ax.legend(handles=legend_handles([m for m in METHODS if m in results]), **kw)
        legend.set_zorder(10)
        style_legend_frame(legend)
    set_budget_xticks(ax, budget)                                # last: measures the final tick labels

    size = check_panel_fits(fig, ax)
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    stem = FIG_DIR / (out_stem or f"{PROJECT}_return")
    fig.savefig(stem.with_suffix(".pdf"))
    fig.savefig(stem.with_suffix(".png"), dpi=300)
    print(f"saved {stem.with_suffix('.pdf')} ({size[0]:.2f} x {size[1]:.2f} in, plot area "
          f"{plot_area()[0]:.2f} x {plot_area()[1]:.2f} in)")
    return fig, ax


fig, ax = plot_panel(results)
plt.show()


saved figures/carl_vehicle_racing_all_return.pdf (2.64 x 1.70 in, plot area 2.17 x 1.24 in)


## Standalone legends

Same handles as the panel (dashed = reference bounds). The horizontal legend is designed for the
full 5.5 in text width; the vertical one for placing beside or inside a panel.


In [7]:
def save_legend(orientation="horizontal", labels=None, out_stem=None):
    labels = list(labels or METHODS)
    handles = legend_handles(labels)
    if orientation == "horizontal":
        fig = plt.figure(figsize=(5.5, 0.3))
        ncol = len(labels)
    else:
        fig = plt.figure(figsize=(1.0, 0.16 * len(labels) + 0.1))
        ncol = 1
    legend = fig.legend(handles=handles, loc="center", ncol=ncol, frameon=True, fancybox=False,
                        edgecolor=LEGEND_EDGE, facecolor="white", framealpha=1.0,
                        handlelength=2.2, handletextpad=0.6, columnspacing=1.4, borderaxespad=0, borderpad=0.5)
    style_legend_frame(legend)
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    stem = FIG_DIR / (out_stem or f"legend_{orientation}")
    pad = 0.02 + (LEGEND_SHADOW["size"] / 72 if LEGEND_SHADOW else 0)   # inches; keep the shadow inside the crop
    fig.savefig(stem.with_suffix(".pdf"), bbox_inches="tight", pad_inches=pad)
    fig.savefig(stem.with_suffix(".png"), dpi=300, bbox_inches="tight", pad_inches=pad)
    print(f"saved {stem.with_suffix('.pdf')}")
    plt.show()


save_legend("horizontal")
save_legend("vertical")
if EXCLUDE:                                  # this panel's own legend, without the excluded methods
    for orientation in ("horizontal", "vertical"):
        save_legend(orientation, labels=[m for m in METHODS if m not in EXCLUDE],
                    out_stem=f"legend_{PROJECT}_{orientation}")

saved figures/legend_horizontal.pdf
saved figures/legend_vertical.pdf
saved figures/legend_carl_vehicle_racing_all_horizontal.pdf
saved figures/legend_carl_vehicle_racing_all_vertical.pdf


## Optional: per-seed final return (same definition as the workbooks)

Mean of the EMA curve over the last 10% of training (`Final_Return` in the xlsx: steps ≥ 45 000 for
a 50 000-episode run). Handy for sanity-checking against the sweep workbooks.


In [8]:
def final_return_table(results, last_fraction=0.10):
    rows = []
    for label, r in results.items():
        start = r["grid"][-1] * (1 - last_fraction)
        sel = r["grid"] >= start
        for i, curve in enumerate(r["per_seed"]):
            rows.append(dict(method=label, seed_idx=i, final_return=curve[sel].mean()))
    return pd.DataFrame(rows)


final_return_table(results).groupby("method", sort=False)["final_return"].agg(["mean", "std", "count"])


,mean,std,count
method,,,
MATE,730.323455,12.766831,3
GPT-2,715.822298,50.190589,3
LSTM,775.210573,19.594596,3
SplAgger,762.943473,21.200906,3
Mamba,731.866321,2.748843,3
Markov,615.213643,41.378430,3


## LaTeX (2 rows x 3 columns, legend above)

```latex
\begin{figure}[t]
  \centering
  \includegraphics{figures/legend_horizontal.pdf}\\[2pt]
  \includegraphics{figures/cheetah-vel_return.pdf}\hfill
  \includegraphics{figures/ant-dir_return.pdf}\hfill
  \includegraphics{figures/hopper-param_return.pdf}\\[2pt]
  \includegraphics{figures/walker-param_return.pdf}\hfill
  \includegraphics{figures/ML10_return.pdf}\hfill
  \includegraphics{figures/ML45_return.pdf}
  \caption{...}
\end{figure}
```

Generate the panels with `SHOW_YLABEL = True` only for the leftmost environment of each row (and
`ROW_N, N_YLABEL = 3, 1`, `PANEL_H = 1.35`, one `XTICK_RESERVE_RIGHT` for all six), and include every PDF at
its native size: the files differ in width, the plot areas do not.
